In [10]:
!pip install -q kaggle pyyaml

In [12]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

print("Dataset downloaded to:")
print(dataset_path)

100%|██████████| 3.83G/3.83G [00:43<00:00, 95.0MB/s]

Extracting files...


Dataset downloaded to:
/root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5


In [13]:
from pathlib import Path

yaml_files = list(Path(dataset_path).rglob("*.yaml")) + list(Path(dataset_path).rglob("*.yml"))

print("YAML files found:")
for p in yaml_files:
    print(p)

YAML files found:
/root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/military_dataset.yaml


In [14]:
import yaml

if not yaml_files:
    raise FileNotFoundError("No YAML file found in downloaded dataset.")

DATA_YAML = str(yaml_files[0])

print("Using:", DATA_YAML)

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("\nFULL CONFIG:")
print(cfg)

print("\nNUMBER OF CLASSES:")
print(cfg.get("nc"))

print("\nCLASS NAMES:")
names = cfg.get("names")
print(names)

Using: /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/military_dataset.yaml

FULL CONFIG:
{'path': '/kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset', 'test': 'test/images', 'train': 'train/images', 'val': 'val/images', 'names': {0: 'camouflage_soldier', 1: 'weapon', 2: 'military_tank', 3: 'military_truck', 4: 'military_vehicle', 5: 'civilian', 6: 'soldier', 7: 'civilian_vehicle', 8: 'military_artillery', 9: 'trench', 10: 'military_aircraft', 11: 'military_warship'}}

NUMBER OF CLASSES:
None

CLASS NAMES:
{0: 'camouflage_soldier', 1: 'weapon', 2: 'military_tank', 3: 'military_truck', 4: 'military_vehicle', 5: 'civilian', 6: 'soldier', 7: 'civilian_vehicle', 8: 'military_artillery', 9: 'trench', 10: 'military_aircraft', 11: 'military_warship'}


In [15]:
expected_classes = [
    "camouflage_soldier",
    "weapon",
    "military_tank",
    "military_truck",
    "military_vehicle",
    "civilian",
    "soldier",
    "civilian_vehicle",
    "military_artillery",
    "trench",
    "military_aircraft",
    "military_warship",
]

if isinstance(names, dict):
    actual_classes = [names[i] for i in sorted(names)]
else:
    actual_classes = list(names)

print("Expected:", expected_classes)
print("Actual:  ", actual_classes)
print()
print("EXACT CLASS MATCH:", actual_classes == expected_classes)

Expected: ['camouflage_soldier', 'weapon', 'military_tank', 'military_truck', 'military_vehicle', 'civilian', 'soldier', 'civilian_vehicle', 'military_artillery', 'trench', 'military_aircraft', 'military_warship']
Actual:   ['camouflage_soldier', 'weapon', 'military_tank', 'military_truck', 'military_vehicle', 'civilian', 'soldier', 'civilian_vehicle', 'military_artillery', 'trench', 'military_aircraft', 'military_warship']

EXACT CLASS MATCH: True


In [16]:
from pathlib import Path

root = Path(DATA_YAML).parent

for split in ["train", "val", "valid", "test"]:
    candidates = [
        root / split / "images",
        root / "images" / split,
    ]

    for folder in candidates:
        if folder.exists():
            images = [
                p for p in folder.iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            ]
            print(f"{split}: {len(images)} images -> {folder}")
            break

train: 21978 images -> /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/train/images
val: 2941 images -> /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/val/images
test: 1396 images -> /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/test/images


In [17]:
from pathlib import Path
import yaml

dataset_root = Path(DATA_YAML).parent

colab_cfg = cfg.copy()
colab_cfg["path"] = str(dataset_root)

COLAB_YAML = "/content/military_dataset_colab.yaml"

with open(COLAB_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(colab_cfg, f, sort_keys=False)

print("Dataset root:", dataset_root)
print("New YAML:", COLAB_YAML)

with open(COLAB_YAML, "r") as f:
    print(f.read())

Dataset root: /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset
New YAML: /content/military_dataset_colab.yaml
path: /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset
test: test/images
train: train/images
val: val/images
names:
  0: camouflage_soldier
  1: weapon
  2: military_tank
  3: military_truck
  4: military_vehicle
  5: civilian
  6: soldier
  7: civilian_vehicle
  8: military_artillery
  9: trench
  10: military_aircraft
  11: military_warship



In [18]:
for split in ["train", "val", "test"]:
    images = dataset_root / split / "images"
    labels = dataset_root / split / "labels"

    print(
        split,
        "| images:", images.exists(),
        "| labels:", labels.exists()
    )

train | images: True | labels: True
val | images: True | labels: True
test | images: True | labels: True


In [19]:
!pip install -q ultralytics==8.4.107

In [4]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.107
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
from google.colab import files

uploaded = files.upload()

Saving military_kaggle_v1.pt to military_kaggle_v1.pt


In [7]:
from pathlib import Path

MODEL_V1 = "/content/military_kaggle_v1.pt"

print("Model exists:", Path(MODEL_V1).exists())
print("Size MB:", round(Path(MODEL_V1).stat().st_size / 1024**2, 2))

Model exists: True
Size MB: 21.46


In [11]:
from ultralytics import YOLO

model_v1 = YOLO(MODEL_V1)

results_v1 = model_v1.val(
    data=COLAB_YAML,
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    plots=True
)

NameError: name 'COLAB_YAML' is not defined

In [16]:
from pathlib import Path

bad_model = Path("/content/military_kaggle_v1.pt")

if bad_model.exists():
    bad_model.unlink()
    print("Bad upload deleted.")

Bad upload deleted.


In [17]:
from google.colab import files

uploaded = files.upload()

Saving military_kaggle_v1.pt to military_kaggle_v1.pt


In [18]:
from pathlib import Path

MODEL_V1 = "/content/military_kaggle_v1.pt"

p = Path(MODEL_V1)

print("Model exists:", p.exists())
print("Size MB:", round(p.stat().st_size / 1024**2, 2))

Model exists: True
Size MB: 21.46


In [19]:
from ultralytics import YOLO

model_v1 = YOLO(MODEL_V1)

print("V1 loaded successfully.")
print(model_v1.names)

V1 loaded successfully.
{0: 'camouflage_soldier', 1: 'weapon', 2: 'military_tank', 3: 'military_truck', 4: 'military_vehicle', 5: 'civilian', 6: 'soldier', 7: 'civilian_vehicle', 8: 'military_artillery', 9: 'trench', 10: 'military_aircraft', 11: 'military_warship'}


In [20]:
results_v1 = model_v1.val(
    data=COLAB_YAML,
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    plots=True
)

Ultralytics 8.4.107 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,130,228 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2203.8±760.2 MB/s, size: 137.4 KB)
val: Scanning /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/val/labels... 2941 images, 273 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2941/2941 2.1Kit/s 1.4s
val: New cache created: /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 368/368 7.7it/s 47.5s
                   all       2941       5081      0.682      0.531      0.567      0.374
    camouflage_soldier        385        510      0.819      0.725      0.806      0.452
                weapon        222        35

In [21]:
v1_baseline = {
    "images": 2941,
    "instances": 5081,
    "precision": 0.682,
    "recall": 0.531,
    "mAP50": 0.567,
    "mAP50_95": 0.374,
    "imgsz": 640,
    "batch": 8,
    "gpu": "Tesla T4"
}

print("V1 BASELINE")
for key, value in v1_baseline.items():
    print(f"{key}: {value}")

V1 BASELINE
images: 2941
instances: 5081
precision: 0.682
recall: 0.531
mAP50: 0.567
mAP50_95: 0.374
imgsz: 640
batch: 8
gpu: Tesla T4


In [22]:
v1_baseline = {
    "precision": float(results_v1.box.mp),
    "recall": float(results_v1.box.mr),
    "mAP50": float(results_v1.box.map50),
    "mAP50_95": float(results_v1.box.map),
}

print("V1 BASELINE")
for k, v in v1_baseline.items():
    print(f"{k}: {v:.4f}")

V1 BASELINE
precision: 0.6819
recall: 0.5314
mAP50: 0.5672
mAP50_95: 0.3744


V2 ARCHITECTURE

In [23]:
import copy
import torch
import torch.nn as nn


def inject_mc_dropout(det_model, p=0.20):
    """
    Insert Dropout2d before the final prediction convolution
    in both box-regression (cv2) and classification (cv3)
    branches at all 3 YOLO detection scales.
    """

    head = det_model.model[-1]

    print("Detection head:", type(head).__name__)

    for branch_name in ["cv2", "cv3"]:
        branches = getattr(head, branch_name)

        for i in range(len(branches)):
            seq = branches[i]
            modules = list(seq.children())

            # Avoid inserting twice
            if any(isinstance(m, nn.Dropout2d) for m in modules):
                print(f"{branch_name}[{i}] already contains dropout")
                continue

            if not isinstance(modules[-1], nn.Conv2d):
                raise TypeError(
                    f"Expected final Conv2d in {branch_name}[{i}], "
                    f"found {type(modules[-1]).__name__}"
                )

            branches[i] = nn.Sequential(
                *modules[:-1],
                nn.Dropout2d(p=p),
                modules[-1]
            )

            print(f"Inserted Dropout2d(p={p}) into {branch_name}[{i}]")

    return det_model


test_v2_model = copy.deepcopy(model_v1.model)

inject_mc_dropout(test_v2_model, p=0.20)

dropouts = [
    m for m in test_v2_model.modules()
    if isinstance(m, nn.Dropout2d)
]

print("\nDropout2d count:", len(dropouts))
print("Probabilities:", [m.p for m in dropouts])

Detection head: Detect
Inserted Dropout2d(p=0.2) into cv2[0]
Inserted Dropout2d(p=0.2) into cv2[1]
Inserted Dropout2d(p=0.2) into cv2[2]
Inserted Dropout2d(p=0.2) into cv3[0]
Inserted Dropout2d(p=0.2) into cv3[1]
Inserted Dropout2d(p=0.2) into cv3[2]

Dropout2d count: 6
Probabilities: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2]


In [24]:
from ultralytics.models.yolo.detect import DetectionTrainer


class MCDODetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):

        model = super().get_model(
            cfg=cfg,
            weights=weights,
            verbose=verbose
        )

        inject_mc_dropout(model, p=0.20)

        dropout_count = sum(
            isinstance(m, nn.Dropout2d)
            for m in model.modules()
        )

        print("\nV2 training model created.")
        print("Dropout2d layers:", dropout_count)

        if dropout_count != 6:
            raise RuntimeError(
                f"Expected 6 Dropout2d layers, found {dropout_count}"
            )

        return model

In [25]:
from ultralytics import YOLO

v2_smoke = YOLO(MODEL_V1)

v2_smoke_results = v2_smoke.train(
    trainer=MCDODetectionTrainer,

    data=COLAB_YAML,

    epochs=1,
    imgsz=640,
    batch=8,

    device=0,
    workers=2,

    project="/content/runs/mcdo_v2",
    name="smoke_p020",
    exist_ok=True,

    seed=0,
    deterministic=True,
    amp=True,

    val=True,
    plots=False
)

New https://pypi.org/project/ultralytics/8.4.131 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.107 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/military_dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

In [3]:
!pip install -q ultralytics==8.4.107

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.6 MB/s eta 0:00:00


In [4]:
import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.107
CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path

V2_SMOKE = "/content/runs/mcdo_v2/smoke_p020/weights/best.pt"

print("Checkpoint exists:", Path(V2_SMOKE).exists())

if Path(V2_SMOKE).exists():
    print(
        "Size MB:",
        round(Path(V2_SMOKE).stat().st_size / 1024**2, 2)
    )

Checkpoint exists: False


In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

V2_DRIVE = Path("/content/drive/MyDrive/UAV_MC_DROPOUT_V2")
V2_DRIVE.mkdir(parents=True, exist_ok=True)

print(V2_DRIVE)

/content/drive/MyDrive/UAV_MC_DROPOUT_V2


In [8]:
!pip install -q ultralytics==8.4.107 kagglehub pyyaml

In [9]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Ultralytics: 8.4.107
CUDA: True
GPU: Tesla T4


In [10]:
import kagglehub
import yaml
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

yaml_files = list(Path(dataset_path).rglob("*.yaml")) + \
             list(Path(dataset_path).rglob("*.yml"))

DATA_YAML = str(yaml_files[0])

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset_root = Path(DATA_YAML).parent

colab_cfg = cfg.copy()
colab_cfg["path"] = str(dataset_root)

COLAB_YAML = "/content/military_dataset_colab.yaml"

with open(COLAB_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(colab_cfg, f, sort_keys=False)

print("Dataset ready:", dataset_root)

100%|██████████| 3.83G/3.83G [00:41<00:00, 98.2MB/s]

Extracting files...


Dataset ready: /root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset


In [11]:
from google.colab import files

uploaded = files.upload()

Saving military_kaggle_v1.pt to military_kaggle_v1.pt


In [22]:
MODEL_V1 = "/content/military_kaggle_v1.pt"

print(
    "Size MB:",
    round(Path(MODEL_V1).stat().st_size / 1024**2, 2)
)

Size MB: 21.46


In [23]:
import torch.nn as nn

def inject_mc_dropout(det_model, p=0.20):
    """
    Insert Dropout2d before the final prediction convolution
    in both the box-regression branch (cv2)
    and classification branch (cv3)
    at all 3 YOLO detection scales.
    """

    head = det_model.model[-1]

    print("Detection head:", type(head).__name__)

    for branch_name in ["cv2", "cv3"]:
        branches = getattr(head, branch_name)

        for i in range(len(branches)):
            seq = branches[i]
            modules = list(seq.children())

            # Avoid inserting dropout twice
            if any(isinstance(m, nn.Dropout2d) for m in modules):
                print(f"{branch_name}[{i}] already contains dropout")
                continue

            if not isinstance(modules[-1], nn.Conv2d):
                raise TypeError(
                    f"Expected final Conv2d in {branch_name}[{i}], "
                    f"found {type(modules[-1]).__name__}"
                )

            branches[i] = nn.Sequential(
                *modules[:-1],
                nn.Dropout2d(p=p),
                modules[-1]
            )

            print(f"Inserted Dropout2d(p={p}) into {branch_name}[{i}]")

    return det_model

In [24]:
from ultralytics.models.yolo.detect import DetectionTrainer

class MCDODetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):

        model = super().get_model(
            cfg=cfg,
            weights=weights,
            verbose=verbose
        )

        inject_mc_dropout(model, p=0.20)

        dropout_count = sum(
            isinstance(m, nn.Dropout2d)
            for m in model.modules()
        )

        print("\nV2 training model created.")
        print("Dropout2d layers:", dropout_count)

        if dropout_count != 6:
            raise RuntimeError(
                f"Expected 6 Dropout2d layers, found {dropout_count}"
            )

        return model

In [25]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

V2_DRIVE = Path("/content/drive/MyDrive/UAV_MC_DROPOUT_V2")
V2_DRIVE.mkdir(parents=True, exist_ok=True)

print("Saving V2 experiments to:", V2_DRIVE)

Mounted at /content/drive
Saving V2 experiments to: /content/drive/MyDrive/UAV_MC_DROPOUT_V2


In [26]:
from pathlib import Path

print("V1 model:", Path(MODEL_V1).exists())
print("Dataset YAML:", Path(COLAB_YAML).exists())
print("Drive folder:", V2_DRIVE.exists())

V1 model: True
Dataset YAML: True
Drive folder: True


In [27]:
from ultralytics import YOLO

v2_smoke = YOLO(MODEL_V1)

v2_smoke_results = v2_smoke.train(
    trainer=MCDODetectionTrainer,

    data=COLAB_YAML,

    epochs=1,
    imgsz=640,
    batch=8,

    device=0,
    workers=2,

    project=str(V2_DRIVE),
    name="smoke_p020",
    exist_ok=True,

    seed=0,
    deterministic=True,
    amp=True,

    val=True,
    plots=False
)

New https://pypi.org/project/ultralytics/8.4.132 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.107 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/military_dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

In [28]:
from ultralytics import YOLO
import torch.nn as nn
from pathlib import Path

V2_SMOKE = "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/smoke_p020/weights/best.pt"

print("Checkpoint exists:", Path(V2_SMOKE).exists())

v2_test = YOLO(V2_SMOKE)

dropouts = [
    m for m in v2_test.model.modules()
    if isinstance(m, nn.Dropout2d)
]

print("V2 checkpoint loaded successfully.")
print("Dropout2d count:", len(dropouts))
print("Dropout probabilities:", [m.p for m in dropouts])

Checkpoint exists: True
V2 checkpoint loaded successfully.
Dropout2d count: 6
Dropout probabilities: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2]


In [29]:
import torch
import torch.nn as nn

core_model = v2_test.model.cuda()

# First put the entire network in inference/evaluation mode
core_model.eval()

# Then reactivate ONLY Dropout2d
for module in core_model.modules():
    if isinstance(module, nn.Dropout2d):
        module.train()

    elif isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

dropout_states = [
    m.training
    for m in core_model.modules()
    if isinstance(m, nn.Dropout2d)
]

bn_states = [
    m.training
    for m in core_model.modules()
    if isinstance(m, nn.modules.batchnorm._BatchNorm)
]

print("Dropout layers:", len(dropout_states))
print("All dropout active:", all(dropout_states))

print("BatchNorm layers:", len(bn_states))
print("All BatchNorm in eval:", not any(bn_states))

Dropout layers: 6
All dropout active: True
BatchNorm layers: 57
All BatchNorm in eval: True


In [30]:
from pathlib import Path

val_images = dataset_root / "val" / "images"

image_files = [
    p for p in val_images.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]

TEST_IMAGE = image_files[0]

print("Test image:")
print(TEST_IMAGE)

Test image:
/root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/val/images/023297.jpg


In [31]:
import cv2
import torch

img = cv2.imread(str(TEST_IMAGE))

if img is None:
    raise RuntimeError("Could not read test image.")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (640, 640))

x = (
    torch.from_numpy(img)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .cuda()
    / 255.0
)

x = x.contiguous()

print("Tensor shape:", x.shape)
print("Device:", x.device)
print("Range:", float(x.min()), "to", float(x.max()))

Tensor shape: torch.Size([1, 3, 640, 640])
Device: cuda:0
Range: 0.0 to 1.0


In [32]:
outputs = []

with torch.no_grad():

    for i in range(10):

        result = core_model(x)

        # YOLO inference can return a tuple.
        if isinstance(result, tuple):
            prediction = result[0]
        else:
            prediction = result

        outputs.append(prediction.detach().cpu())

        print(
            f"Pass {i+1}:",
            "mean =", float(prediction.mean()),
            "std =", float(prediction.std())
        )

Pass 1: mean = 52.64508056640625 std = 127.77517700195312
Pass 2: mean = 52.89042282104492 std = 127.4212875366211
Pass 3: mean = 52.9635124206543 std = 127.5887222290039
Pass 4: mean = 53.027462005615234 std = 127.12615966796875
Pass 5: mean = 53.49265670776367 std = 128.44422912597656
Pass 6: mean = 51.78608703613281 std = 126.6520767211914
Pass 7: mean = 52.01917266845703 std = 126.57893371582031
Pass 8: mean = 53.19770050048828 std = 127.5871810913086
Pass 9: mean = 51.81251525878906 std = 126.82315063476562
Pass 10: mean = 53.427669525146484 std = 128.14559936523438


In [33]:
print("\nDifference from Pass 1:")

reference = outputs[0]

for i, output in enumerate(outputs[1:], start=2):

    difference = torch.mean(
        torch.abs(output - reference)
    ).item()

    print(
        f"Pass {i} vs Pass 1:",
        difference
    )


Difference from Pass 1:
Pass 2 vs Pass 1: 1.5516228675842285
Pass 3 vs Pass 1: 2.0108327865600586
Pass 4 vs Pass 1: 2.1672651767730713
Pass 5 vs Pass 1: 2.1107211112976074
Pass 6 vs Pass 1: 1.5460965633392334
Pass 7 vs Pass 1: 1.778049349784851
Pass 8 vs Pass 1: 1.6828486919403076
Pass 9 vs Pass 1: 1.5251717567443848
Pass 10 vs Pass 1: 1.5637797117233276


In [34]:
import json
import shutil
from pathlib import Path
import torch

SAVE_DIR = V2_DRIVE / "session_results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save the Colab-compatible dataset configuration
if Path(COLAB_YAML).exists():
    shutil.copy2(
        COLAB_YAML,
        SAVE_DIR / "military_dataset_colab.yaml"
    )

# Save the exact test image used
if "TEST_IMAGE" in globals() and Path(TEST_IMAGE).exists():
    shutil.copy2(
        TEST_IMAGE,
        SAVE_DIR / Path(TEST_IMAGE).name
    )

# Calculate the stochastic differences again
mc_differences = []

if "outputs" in globals() and len(outputs) > 1:
    reference = outputs[0]

    for i, output in enumerate(outputs[1:], start=2):
        difference = torch.mean(
            torch.abs(output - reference)
        ).item()

        mc_differences.append({
            "pass": i,
            "difference_vs_pass_1": difference
        })

summary = {
    "method": "MC Dropout V2 smoke test",
    "dropout_type": "Dropout2d",
    "dropout_probability": 0.20,
    "dropout_layers": 6,
    "mc_passes": len(outputs) if "outputs" in globals() else None,

    "v1_baseline": {
        "precision": 0.682,
        "recall": 0.531,
        "mAP50": 0.567,
        "mAP50_95": 0.374
    },

    "v2_smoke_validation": {
        "precision": 0.548,
        "recall": 0.437,
        "mAP50": 0.439,
        "mAP50_95": 0.268
    },

    "mc_stochasticity_confirmed": (
        len(mc_differences) > 0 and
        any(x["difference_vs_pass_1"] > 0 for x in mc_differences)
    ),

    "mc_differences": mc_differences,

    "checkpoint": str(
        V2_DRIVE / "smoke_p020" / "weights" / "best.pt"
    )
}

SUMMARY_FILE = SAVE_DIR / "v2_smoke_test_summary.json"

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("EVERYTHING IMPORTANT SAVED")
print("Summary:", SUMMARY_FILE)
print("Model:", summary["checkpoint"])
print("MC stochasticity confirmed:",
      summary["mc_stochasticity_confirmed"])

EVERYTHING IMPORTANT SAVED
Summary: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/session_results/v2_smoke_test_summary.json
Model: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/smoke_p020/weights/best.pt
MC stochasticity confirmed: True


In [1]:
!pip install -q ultralytics==8.4.107 kagglehub pyyaml

from google.colab import drive
drive.mount("/content/drive")

import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.6 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.107
CUDA: True
GPU: Tesla T4


In [2]:
import kagglehub
import yaml
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

yaml_files = list(Path(dataset_path).rglob("*.yaml")) + \
             list(Path(dataset_path).rglob("*.yml"))

DATA_YAML = str(yaml_files[0])

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset_root = Path(DATA_YAML).parent

colab_cfg = cfg.copy()
colab_cfg["path"] = str(dataset_root)

COLAB_YAML = "/content/military_dataset_colab.yaml"

with open(COLAB_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(colab_cfg, f, sort_keys=False)

print("Dataset ready:", dataset_root)

Using Colab cache for faster access to the 'military-assets-dataset-12-classes-yolo8-format' dataset.
Dataset ready: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset


In [3]:
from pathlib import Path

V2_DRIVE = Path("/content/drive/MyDrive/UAV_MC_DROPOUT_V2")
BASELINE_DIR = V2_DRIVE / "baseline"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

V1_DRIVE = BASELINE_DIR / "military_kaggle_v1.pt"

print("V1 already saved:", V1_DRIVE.exists())

V1 already saved: False


In [4]:
from google.colab import files
import shutil

uploaded = files.upload()

uploaded_name = next(iter(uploaded))
uploaded_path = Path("/content") / uploaded_name

print(
    "Uploaded size MB:",
    round(uploaded_path.stat().st_size / 1024**2, 2)
)

shutil.copy2(uploaded_path, V1_DRIVE)

print("Permanent V1:", V1_DRIVE)

Saving military_kaggle_v1.pt to military_kaggle_v1.pt
Uploaded size MB: 21.46
Permanent V1: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/baseline/military_kaggle_v1.pt


In [5]:
MODEL_V1 = str(V1_DRIVE)

print("Using V1:", MODEL_V1)
print("Exists:", Path(MODEL_V1).exists())

Using V1: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/baseline/military_kaggle_v1.pt
Exists: True


In [6]:
import torch.nn as nn

def inject_mc_dropout(det_model, p=0.20):

    head = det_model.model[-1]

    print("Detection head:", type(head).__name__)

    for branch_name in ["cv2", "cv3"]:
        branches = getattr(head, branch_name)

        for i in range(len(branches)):
            seq = branches[i]
            modules = list(seq.children())

            if any(isinstance(m, nn.Dropout2d) for m in modules):
                continue

            if not isinstance(modules[-1], nn.Conv2d):
                raise TypeError(
                    f"Expected final Conv2d in {branch_name}[{i}]"
                )

            branches[i] = nn.Sequential(
                *modules[:-1],
                nn.Dropout2d(p=p),
                modules[-1]
            )

            print(
                f"Inserted Dropout2d(p={p}) "
                f"into {branch_name}[{i}]"
            )

    return det_model

In [7]:
from ultralytics.models.yolo.detect import DetectionTrainer

class MCDODetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):

        model = super().get_model(
            cfg=cfg,
            weights=weights,
            verbose=verbose
        )

        inject_mc_dropout(model, p=0.20)

        dropout_count = sum(
            isinstance(m, nn.Dropout2d)
            for m in model.modules()
        )

        print("\nV2 model created.")
        print("Dropout2d layers:", dropout_count)

        if dropout_count != 6:
            raise RuntimeError(
                f"Expected 6 dropout layers, "
                f"found {dropout_count}"
            )

        return model

In [8]:
from ultralytics import YOLO

v2_head = YOLO(MODEL_V1)

v2_head_results = v2_head.train(
    trainer=MCDODetectionTrainer,

    data=COLAB_YAML,

    epochs=3,
    imgsz=640,
    batch=8,

    device=0,
    workers=2,

    # Freeze model layers 0-21.
    # Detect head (22) remains trainable.
    freeze=22,

    # Conservative optimizer setup
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.1,
    weight_decay=0.0005,

    warmup_epochs=1.0,
    cos_lr=True,

    # Less aggressive augmentation for this adaptation step
    mosaic=0.0,

    seed=0,
    deterministic=True,
    amp=True,

    val=True,
    plots=True,

    project=str(V2_DRIVE),
    name="headonly_p020_lr1e4_e3",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.135 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.107 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/military_dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=tr

In [9]:
from ultralytics import YOLO
import torch.nn as nn
from pathlib import Path

V2_FINAL = (
    "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/"
    "headonly_p020_lr1e4_e3/weights/best.pt"
)

print("Exists:", Path(V2_FINAL).exists())

v2_final = YOLO(V2_FINAL)

dropouts = [
    m for m in v2_final.model.modules()
    if isinstance(m, nn.Dropout2d)
]

print("Dropout2d count:", len(dropouts))
print("Probabilities:", [m.p for m in dropouts])

Exists: True
Dropout2d count: 6
Probabilities: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2]


In [10]:
import torch
import torch.nn as nn

core_model = v2_final.model.cuda()

# Entire network to evaluation mode
core_model.eval()

# Reactivate ONLY dropout
for module in core_model.modules():

    if isinstance(module, nn.Dropout2d):
        module.train()

    elif isinstance(
        module,
        nn.modules.batchnorm._BatchNorm
    ):
        module.eval()

dropout_states = [
    m.training
    for m in core_model.modules()
    if isinstance(m, nn.Dropout2d)
]

bn_states = [
    m.training
    for m in core_model.modules()
    if isinstance(
        m,
        nn.modules.batchnorm._BatchNorm
    )
]

print("Dropout count:", len(dropout_states))
print("All dropout ACTIVE:", all(dropout_states))

print("BatchNorm count:", len(bn_states))
print("All BatchNorm EVAL:", not any(bn_states))

Dropout count: 6
All dropout ACTIVE: True
BatchNorm count: 57
All BatchNorm EVAL: True


In [13]:
from pathlib import Path
import cv2
import torch

# Pick one validation image
val_images = dataset_root / "val" / "images"

image_files = [
    p for p in val_images.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]

TEST_IMAGE = image_files[0]

print("Using SAME test image:")
print(TEST_IMAGE)

# Load image
img = cv2.imread(str(TEST_IMAGE))

if img is None:
    raise RuntimeError("Could not read test image.")

# Convert BGR -> RGB
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Resize to YOLO input size
img = cv2.resize(img, (640, 640))

# Convert image into PyTorch tensor
x = (
    torch.from_numpy(img)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .cuda()
    / 255.0
)

x = x.contiguous()

print("Tensor shape:", x.shape)
print("Device:", x.device)
print("Pixel range:", float(x.min()), "to", float(x.max()))

Using SAME test image:
/root/.cache/kagglehub/datasets/rawsi18/military-assets-dataset-12-classes-yolo8-format/versions/5/military_object_dataset/val/images/023297.jpg
Tensor shape: torch.Size([1, 3, 640, 640])
Device: cuda:0
Pixel range: 0.0 to 1.0


In [14]:
outputs_final = []

with torch.no_grad():

    for i in range(20):

        result = core_model(x)

        if isinstance(result, tuple):
            prediction = result[0]
        else:
            prediction = result

        outputs_final.append(
            prediction.detach().cpu()
        )

print("Completed:", len(outputs_final), "MC passes")

Completed: 20 MC passes


In [15]:
reference = outputs_final[0]

print("\nDifferences from Pass 1:")

differences = []

for i, output in enumerate(outputs_final[1:], start=2):

    difference = torch.mean(
        torch.abs(output - reference)
    ).item()

    differences.append(difference)

    print(
        f"Pass {i:02d} vs Pass 1: {difference:.6f}"
    )

print("\nMean difference:", sum(differences) / len(differences))
print("Minimum difference:", min(differences))
print("Maximum difference:", max(differences))
print(
    "MC stochasticity confirmed:",
    any(d > 0 for d in differences)
)


Differences from Pass 1:
Pass 02 vs Pass 1: 1.417507
Pass 03 vs Pass 1: 1.285586
Pass 04 vs Pass 1: 1.466128
Pass 05 vs Pass 1: 1.160171
Pass 06 vs Pass 1: 1.513335
Pass 07 vs Pass 1: 1.488296
Pass 08 vs Pass 1: 1.222568
Pass 09 vs Pass 1: 1.346626
Pass 10 vs Pass 1: 1.563683
Pass 11 vs Pass 1: 1.535272
Pass 12 vs Pass 1: 1.137756
Pass 13 vs Pass 1: 1.314813
Pass 14 vs Pass 1: 1.276106
Pass 15 vs Pass 1: 1.463266
Pass 16 vs Pass 1: 1.275758
Pass 17 vs Pass 1: 1.454262
Pass 18 vs Pass 1: 1.301034
Pass 19 vs Pass 1: 1.479830
Pass 20 vs Pass 1: 1.271558

Mean difference: 1.3670291963376497
Minimum difference: 1.137756109237671
Maximum difference: 1.563683271408081
MC stochasticity confirmed: True


In [16]:
import json
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/"
    "headonly_p020_lr1e4_e3/verification"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

verification = {
    "model": "headonly_p020_lr1e4_e3",
    "dropout_type": "Dropout2d",
    "dropout_probability": 0.20,
    "dropout_layers": 6,
    "batchnorm_layers": 57,
    "dropout_active": True,
    "batchnorm_eval": True,
    "mc_passes": 20,
    "mean_raw_output_difference": 1.3670291963376497,
    "min_raw_output_difference": 1.137756109237671,
    "max_raw_output_difference": 1.563683271408081,
    "stochasticity_confirmed": True,

    "v1_baseline": {
        "precision": 0.682,
        "recall": 0.531,
        "mAP50": 0.567,
        "mAP50_95": 0.374
    },

    "v2_validation": {
        "precision": 0.678,
        "recall": 0.523,
        "mAP50": 0.574,
        "mAP50_95": 0.375
    }
}

out_file = RESULT_DIR / "mcdo_v2_verification.json"

with open(out_file, "w") as f:
    json.dump(verification, f, indent=2)

print("Saved:", out_file)

Saved: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/headonly_p020_lr1e4_e3/verification/mcdo_v2_verification.json


In [3]:
!pip install -q ultralytics==8.4.107 kagglehub pyyaml

from google.colab import drive
drive.mount("/content/drive")

import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics: 8.4.107
CUDA: True
GPU: Tesla T4


In [4]:
from ultralytics import YOLO
import torch.nn as nn
from pathlib import Path

V2_FINAL = (
    "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/"
    "headonly_p020_lr1e4_e3/weights/best.pt"
)

print("Model exists:", Path(V2_FINAL).exists())

v2_final = YOLO(V2_FINAL)

dropouts = [
    m for m in v2_final.model.modules()
    if isinstance(m, nn.Dropout2d)
]

print("Dropout layers:", len(dropouts))
print("Probabilities:", [m.p for m in dropouts])

Model exists: True
Dropout layers: 6
Probabilities: [0.2, 0.2, 0.2, 0.2, 0.2, 0.2]


In [5]:
import torch
import torch.nn as nn

core_model = v2_final.model.cuda()

# Everything starts in inference mode
core_model.eval()

# Reactivate ONLY dropout
for module in core_model.modules():

    if isinstance(module, nn.Dropout2d):
        module.train()

    elif isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

dropout_states = [
    m.training
    for m in core_model.modules()
    if isinstance(m, nn.Dropout2d)
]

bn_states = [
    m.training
    for m in core_model.modules()
    if isinstance(m, nn.modules.batchnorm._BatchNorm)
]

print("Dropout ACTIVE:", all(dropout_states))
print("BatchNorm EVAL:", not any(bn_states))

Dropout ACTIVE: True
BatchNorm EVAL: True


In [6]:
import kagglehub
import yaml
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

yaml_files = (
    list(Path(dataset_path).rglob("*.yaml"))
    + list(Path(dataset_path).rglob("*.yml"))
)

DATA_YAML = str(yaml_files[0])
dataset_root = Path(DATA_YAML).parent

print("Dataset:", dataset_root)

Using Colab cache for faster access to the 'military-assets-dataset-12-classes-yolo8-format' dataset.
Dataset: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset


In [7]:
import kagglehub
import yaml
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

yaml_files = (
    list(Path(dataset_path).rglob("*.yaml"))
    + list(Path(dataset_path).rglob("*.yml"))
)

DATA_YAML = str(yaml_files[0])
dataset_root = Path(DATA_YAML).parent

print("Dataset:", dataset_root)

Using Colab cache for faster access to the 'military-assets-dataset-12-classes-yolo8-format' dataset.
Dataset: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset


In [8]:
import kagglehub
import yaml
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "rawsi18/military-assets-dataset-12-classes-yolo8-format"
)

yaml_files = (
    list(Path(dataset_path).rglob("*.yaml"))
    + list(Path(dataset_path).rglob("*.yml"))
)

DATA_YAML = str(yaml_files[0])
dataset_root = Path(DATA_YAML).parent

print("Dataset:", dataset_root)

Using Colab cache for faster access to the 'military-assets-dataset-12-classes-yolo8-format' dataset.
Dataset: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset


In [9]:
from pathlib import Path

val_images = dataset_root / "val" / "images"
val_labels = dataset_root / "val" / "labels"

TANK_CLASS_ID = 2

TEST_IMAGE = None

for label_file in val_labels.glob("*.txt"):

    lines = label_file.read_text().strip().splitlines()

    if any(
        line.strip()
        and int(float(line.split()[0])) == TANK_CLASS_ID
        for line in lines
    ):
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            candidate = val_images / (label_file.stem + ext)

            if candidate.exists():
                TEST_IMAGE = candidate
                break

    if TEST_IMAGE is not None:
        break

print("Selected image:")
print(TEST_IMAGE)

Selected image:
/kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset/val/images/013256.jpg


In [11]:
import cv2
import numpy as np
import torch

from ultralytics.utils.nms import non_max_suppression
from ultralytics.data.augment import LetterBox

N_MC = 20
CONF_THRES = 0.25
NMS_IOU = 0.45
IMGSZ = 640
# Load original image
original = cv2.imread(str(TEST_IMAGE))

if original is None:
    raise RuntimeError("Could not read image.")

h0, w0 = original.shape[:2]

# YOLO-style letterbox resize
letterbox = LetterBox(
    new_shape=(IMGSZ, IMGSZ),
    auto=False,
    stride=32
)

processed = letterbox(image=original)

# BGR -> RGB
processed = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

x = (
    torch.from_numpy(processed)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .cuda()
    / 255.0
)

x = x.contiguous()

mc_detections = []

with torch.no_grad():

    for pass_idx in range(N_MC):

        raw = core_model(x)

        if isinstance(raw, tuple):
            raw = raw[0]

        detections = non_max_suppression(
            raw,
            conf_thres=CONF_THRES,
            iou_thres=NMS_IOU,
            nc=12
        )[0]

        pass_results = []

        if detections is not None and len(detections):

            for det in detections.cpu().numpy():

                x1, y1, x2, y2, conf, cls_id = det[:6]

                pass_results.append({
                    "box": np.array(
                        [x1, y1, x2, y2],
                        dtype=float
                    ),
                    "confidence": float(conf),
                    "class_id": int(cls_id),
                    "class_name": v2_final.names[int(cls_id)]
                })

        mc_detections.append(pass_results)

        print(
            f"Pass {pass_idx + 1:02d}: "
            f"{len(pass_results)} detections"
        )

Pass 01: 1 detections
Pass 02: 1 detections
Pass 03: 1 detections
Pass 04: 1 detections
Pass 05: 1 detections
Pass 06: 1 detections
Pass 07: 1 detections
Pass 08: 1 detections
Pass 09: 1 detections
Pass 10: 1 detections
Pass 11: 1 detections
Pass 12: 1 detections
Pass 13: 1 detections
Pass 14: 1 detections
Pass 15: 1 detections
Pass 16: 1 detections
Pass 17: 1 detections
Pass 18: 1 detections
Pass 19: 1 detections
Pass 20: 1 detections


In [12]:
for i, detections in enumerate(mc_detections, start=1):

    print(f"\nPASS {i:02d}")

    for det in detections:
        print(
            f'Class: {det["class_name"]:<20} '
            f'Conf: {det["confidence"]:.4f} '
            f'Box: {np.round(det["box"], 1)}'
        )


PASS 01
Class: military_tank        Conf: 0.8583 Box: [      156.9       217.8       488.6       435.7]

PASS 02
Class: military_tank        Conf: 0.8439 Box: [      146.6       216.3       459.8       436.7]

PASS 03
Class: military_tank        Conf: 0.8015 Box: [      153.4         219       458.9       438.1]

PASS 04
Class: military_tank        Conf: 0.6855 Box: [      164.5       220.8       461.5       437.1]

PASS 05
Class: military_tank        Conf: 0.7265 Box: [      177.3         217       463.1       442.3]

PASS 06
Class: military_tank        Conf: 0.8304 Box: [      158.7       228.2         458       436.1]

PASS 07
Class: military_tank        Conf: 0.7895 Box: [      159.9         214       462.5       437.2]

PASS 08
Class: military_tank        Conf: 0.7980 Box: [      164.7       224.6       464.9       436.7]

PASS 09
Class: military_tank        Conf: 0.9003 Box: [      158.1       223.6       462.1       440.4]

PASS 10
Class: military_tank        Conf: 0.8602 Box: 

In [13]:
from collections import Counter
import numpy as np
import math

# One detection per MC pass
detections_1object = [
    pass_detections[0]
    for pass_detections in mc_detections
    if len(pass_detections) == 1
]

boxes = np.array([
    d["box"]
    for d in detections_1object
])

confidences = np.array([
    d["confidence"]
    for d in detections_1object
])

class_ids = [
    d["class_id"]
    for d in detections_1object
]

class_names = [
    d["class_name"]
    for d in detections_1object
]

# ----------------------------
# Persistence
# ----------------------------
detected_count = len(detections_1object)
persistence = detected_count / N_MC

# ----------------------------
# Class statistics
# ----------------------------
class_counts = Counter(class_ids)

dominant_class_id, dominant_count = \
    class_counts.most_common(1)[0]

dominant_class = v2_final.names[dominant_class_id]

class_agreement = dominant_count / detected_count

entropy = 0.0

for count in class_counts.values():
    p = count / detected_count
    entropy -= p * math.log2(p)

# ----------------------------
# Bounding-box statistics
# ----------------------------
centers_x = (boxes[:, 0] + boxes[:, 2]) / 2
centers_y = (boxes[:, 1] + boxes[:, 3]) / 2

widths = boxes[:, 2] - boxes[:, 0]
heights = boxes[:, 3] - boxes[:, 1]

mean_box = boxes.mean(axis=0)

def box_iou(a, b):

    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)

    intersection = inter_w * inter_h

    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])

    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0

ious = [
    box_iou(box, mean_box)
    for box in boxes
]

# ----------------------------
# Results
# ----------------------------

print("\n" + "=" * 55)
print("V2 MC-DROPOUT DETECTION UNCERTAINTY")
print("=" * 55)

print("Dominant class:       ", dominant_class)

print(
    f"Detected:              "
    f"{detected_count}/{N_MC}"
)

print(
    f"Persistence:           "
    f"{persistence:.3f}"
)

print(
    f"Mean confidence:       "
    f"{confidences.mean():.4f}"
)

print(
    f"Confidence std:        "
    f"{confidences.std():.4f}"
)

print(
    f"Confidence min/max:    "
    f"{confidences.min():.4f} / "
    f"{confidences.max():.4f}"
)

print(
    f"Class agreement:       "
    f"{class_agreement:.3f}"
)

print(
    f"Class entropy:         "
    f"{entropy:.4f} bits"
)

print(
    f"Center X std:          "
    f"{centers_x.std():.2f} px"
)

print(
    f"Center Y std:          "
    f"{centers_y.std():.2f} px"
)

print(
    f"Width std:             "
    f"{widths.std():.2f} px"
)

print(
    f"Height std:            "
    f"{heights.std():.2f} px"
)

print(
    f"Mean IoU:              "
    f"{np.mean(ious):.4f}"
)

print(
    f"Minimum IoU:           "
    f"{np.min(ious):.4f}"
)

print("\nClass distribution:")

for cls_id, count in class_counts.items():
    print(
        f"  {v2_final.names[cls_id]}: "
        f"{count}/{N_MC}"
    )


V2 MC-DROPOUT DETECTION UNCERTAINTY
Dominant class:        military_tank
Detected:              20/20
Persistence:           1.000
Mean confidence:       0.8002
Confidence std:        0.0710
Confidence min/max:    0.5847 / 0.9003
Class agreement:       1.000
Class entropy:         0.0000 bits
Center X std:          6.11 px
Center Y std:          1.89 px
Width std:             11.32 px
Height std:            5.22 px
Mean IoU:              0.9444
Minimum IoU:           0.8794

Class distribution:
  military_tank: 20/20


In [14]:
from pathlib import Path
import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression

N_MC_SEARCH = 10
CONF_THRES = 0.25
NMS_IOU = 0.45
IMGSZ = 640
TANK_CLASS_ID = 2

candidate_images = []

# Collect validation images whose ground truth contains a tank
for label_file in val_labels.glob("*.txt"):

    lines = label_file.read_text().strip().splitlines()

    has_tank = any(
        line.strip()
        and int(float(line.split()[0])) == TANK_CLASS_ID
        for line in lines
    )

    if not has_tank:
        continue

    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        candidate = val_images / (label_file.stem + ext)

        if candidate.exists():
            candidate_images.append(candidate)
            break

print("Tank candidate images:", len(candidate_images))

Tank candidate images: 938


In [15]:
def mc_detect_image(image_path, n_passes=10):

    original = cv2.imread(str(image_path))

    if original is None:
        return None

    letterbox = LetterBox(
        new_shape=(IMGSZ, IMGSZ),
        auto=False,
        stride=32
    )

    processed = letterbox(image=original)
    processed = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

    x_local = (
        torch.from_numpy(processed)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .float()
        .cuda()
        / 255.0
    ).contiguous()

    results = []

    with torch.no_grad():

        for _ in range(n_passes):

            raw = core_model(x_local)

            if isinstance(raw, tuple):
                raw = raw[0]

            dets = non_max_suppression(
                raw,
                conf_thres=CONF_THRES,
                iou_thres=NMS_IOU,
                nc=12
            )[0]

            pass_results = []

            if dets is not None and len(dets):

                for det in dets.cpu().numpy():

                    x1, y1, x2, y2, conf, cls_id = det[:6]

                    pass_results.append({
                        "box": np.array(
                            [x1, y1, x2, y2],
                            dtype=float
                        ),
                        "confidence": float(conf),
                        "class_id": int(cls_id),
                        "class_name": v2_final.names[int(cls_id)]
                    })

            results.append(pass_results)

    return results

In [16]:
DIFFICULT_IMAGE = None
DIFFICULT_RESULTS = None

for idx, image_path in enumerate(candidate_images[:100]):

    results = mc_detect_image(
        image_path,
        n_passes=N_MC_SEARCH
    )

    # Count passes containing at least one tank
    tank_passes = 0
    tank_confidences = []

    for pass_results in results:

        tanks = [
            d for d in pass_results
            if d["class_id"] == TANK_CLASS_ID
        ]

        if tanks:
            tank_passes += 1

            # highest-confidence tank in that pass
            best = max(
                tanks,
                key=lambda d: d["confidence"]
            )

            tank_confidences.append(
                best["confidence"]
            )

    persistence = tank_passes / N_MC_SEARCH

    conf_std = (
        float(np.std(tank_confidences))
        if len(tank_confidences) > 1
        else 0.0
    )

    print(
        f"{idx+1:03d} | "
        f"persistence={persistence:.2f} | "
        f"conf_std={conf_std:.3f} | "
        f"{image_path.name}"
    )

    # Look for something clearly less stable
    if persistence < 0.9 or conf_std > 0.10:

        DIFFICULT_IMAGE = image_path
        DIFFICULT_RESULTS = results
        break

print("\nSelected difficult image:")
print(DIFFICULT_IMAGE)

001 | persistence=1.00 | conf_std=0.065 | 013256.jpg
002 | persistence=1.00 | conf_std=0.112 | 008516.jpg

Selected difficult image:
/kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset/val/images/008516.jpg


In [17]:
DIFFICULT_IMAGE = Path(
    "/kaggle/input/military-assets-dataset-12-classes-yolo8-format/"
    "military_object_dataset/val/images/008516.jpg"
)

difficult_mc = mc_detect_image(
    DIFFICULT_IMAGE,
    n_passes=20
)

print("20-pass test complete.")

for i, detections in enumerate(difficult_mc, start=1):
    tanks = [
        d for d in detections
        if d["class_id"] == TANK_CLASS_ID
    ]

    print(
        f"Pass {i:02d}: "
        f"{len(detections)} total detections, "
        f"{len(tanks)} tank detections"
    )

20-pass test complete.
Pass 01: 1 total detections, 1 tank detections
Pass 02: 1 total detections, 1 tank detections
Pass 03: 1 total detections, 1 tank detections
Pass 04: 1 total detections, 1 tank detections
Pass 05: 1 total detections, 1 tank detections
Pass 06: 1 total detections, 1 tank detections
Pass 07: 1 total detections, 1 tank detections
Pass 08: 1 total detections, 1 tank detections
Pass 09: 1 total detections, 1 tank detections
Pass 10: 1 total detections, 1 tank detections
Pass 11: 1 total detections, 1 tank detections
Pass 12: 1 total detections, 1 tank detections
Pass 13: 1 total detections, 1 tank detections
Pass 14: 1 total detections, 1 tank detections
Pass 15: 1 total detections, 1 tank detections
Pass 16: 1 total detections, 1 tank detections
Pass 17: 1 total detections, 1 tank detections
Pass 18: 1 total detections, 1 tank detections
Pass 19: 1 total detections, 1 tank detections
Pass 20: 1 total detections, 1 tank detections


In [18]:
from collections import Counter
import numpy as np
import math

N_MC = 20

# For this screening comparison:
# select the highest-confidence tank in each pass.
tank_detections = []

for pass_idx, detections in enumerate(difficult_mc):

    tanks = [
        d for d in detections
        if d["class_id"] == TANK_CLASS_ID
    ]

    if tanks:
        best_tank = max(
            tanks,
            key=lambda d: d["confidence"]
        )

        tank_detections.append({
            **best_tank,
            "pass_idx": pass_idx
        })

detected_count = len(tank_detections)
persistence = detected_count / N_MC

boxes = np.array([
    d["box"]
    for d in tank_detections
])

confidences = np.array([
    d["confidence"]
    for d in tank_detections
])

class_ids = [
    d["class_id"]
    for d in tank_detections
]

class_counts = Counter(class_ids)

dominant_class_id, dominant_count = \
    class_counts.most_common(1)[0]

class_agreement = dominant_count / detected_count

entropy = 0.0

for count in class_counts.values():
    p = count / detected_count
    entropy -= p * math.log2(p)

centers_x = (boxes[:, 0] + boxes[:, 2]) / 2
centers_y = (boxes[:, 1] + boxes[:, 3]) / 2

widths = boxes[:, 2] - boxes[:, 0]
heights = boxes[:, 3] - boxes[:, 1]

mean_box = boxes.mean(axis=0)

ious = [
    box_iou(box, mean_box)
    for box in boxes
]

print("\n" + "=" * 55)
print("DIFFICULT IMAGE — V2 MC-DROPOUT")
print("=" * 55)

print("Image:                 ", DIFFICULT_IMAGE.name)
print("Dominant class:        ", v2_final.names[dominant_class_id])
print(f"Detected:               {detected_count}/{N_MC}")
print(f"Persistence:            {persistence:.3f}")

print(f"Mean confidence:        {confidences.mean():.4f}")
print(f"Confidence std:         {confidences.std():.4f}")
print(
    f"Confidence min/max:     "
    f"{confidences.min():.4f} / {confidences.max():.4f}"
)

print(f"Class agreement:        {class_agreement:.3f}")
print(f"Class entropy:          {entropy:.4f} bits")

print(f"Center X std:           {centers_x.std():.2f} px")
print(f"Center Y std:           {centers_y.std():.2f} px")

print(f"Width std:              {widths.std():.2f} px")
print(f"Height std:             {heights.std():.2f} px")

print(f"Mean IoU:               {np.mean(ious):.4f}")
print(f"Minimum IoU:            {np.min(ious):.4f}")


DIFFICULT IMAGE — V2 MC-DROPOUT
Image:                  008516.jpg
Dominant class:         military_tank
Detected:               20/20
Persistence:            1.000
Mean confidence:        0.7917
Confidence std:         0.0678
Confidence min/max:     0.6498 / 0.9141
Class agreement:        1.000
Class entropy:          0.0000 bits
Center X std:           3.38 px
Center Y std:           3.06 px
Width std:              7.81 px
Height std:             6.87 px
Mean IoU:               0.9191
Minimum IoU:            0.7571


In [19]:
from pathlib import Path
import cv2
import numpy as np

IMAGE_PATH = DIFFICULT_IMAGE

LABEL_PATH = (
    val_labels /
    f"{IMAGE_PATH.stem}.txt"
)

image_original = cv2.imread(str(IMAGE_PATH))

if image_original is None:
    raise RuntimeError("Could not load image.")

orig_h, orig_w = image_original.shape[:2]

tank_gt_boxes = []

for line in LABEL_PATH.read_text().strip().splitlines():

    values = line.split()

    cls_id = int(float(values[0]))

    if cls_id != TANK_CLASS_ID:
        continue

    xc, yc, w, h = map(float, values[1:5])

    # Normalized YOLO coordinates -> original pixel coordinates
    xc *= orig_w
    yc *= orig_h
    w *= orig_w
    h *= orig_h

    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2

    tank_gt_boxes.append(
        np.array([x1, y1, x2, y2], dtype=float)
    )

print("Ground-truth tanks:", len(tank_gt_boxes))

for box in tank_gt_boxes:
    print(box)

Ground-truth tanks: 1
[     442.22      191.11      796.66      401.11]


In [20]:
IMGSZ = 640

scale = min(
    IMGSZ / orig_w,
    IMGSZ / orig_h
)

new_w = round(orig_w * scale)
new_h = round(orig_h * scale)

pad_x = (IMGSZ - new_w) / 2
pad_y = (IMGSZ - new_h) / 2

def original_to_letterbox(box):

    x1, y1, x2, y2 = box

    return np.array([
        x1 * scale + pad_x,
        y1 * scale + pad_y,
        x2 * scale + pad_x,
        y2 * scale + pad_y
    ], dtype=float)


GT_BOX = original_to_letterbox(
    tank_gt_boxes[0]
)

print("GT box in 640x640 space:")
print(np.round(GT_BOX, 2))

GT box in 640x640 space:
[     223.38      236.54      402.42      342.61]


In [21]:
MATCH_IOU = 0.50

matched_detections = []

for pass_idx, detections in enumerate(difficult_mc):

    best_detection = None
    best_iou = 0.0

    for det in detections:

        iou = box_iou(
            det["box"],
            GT_BOX
        )

        if iou > best_iou:
            best_iou = iou
            best_detection = det

    if (
        best_detection is not None
        and best_iou >= MATCH_IOU
    ):
        matched_detections.append({
            **best_detection,
            "pass_idx": pass_idx,
            "gt_iou": best_iou
        })

        print(
            f"Pass {pass_idx+1:02d}: "
            f'{best_detection["class_name"]:<20} '
            f'conf={best_detection["confidence"]:.3f} '
            f'GT-IoU={best_iou:.3f}'
        )

    else:
        print(
            f"Pass {pass_idx+1:02d}: "
            "MISSED"
        )

Pass 01: military_tank        conf=0.755 GT-IoU=0.911
Pass 02: military_tank        conf=0.755 GT-IoU=0.876
Pass 03: military_tank        conf=0.691 GT-IoU=0.776
Pass 04: military_tank        conf=0.650 GT-IoU=0.752
Pass 05: military_tank        conf=0.885 GT-IoU=0.851
Pass 06: military_tank        conf=0.710 GT-IoU=0.917
Pass 07: military_tank        conf=0.699 GT-IoU=0.893
Pass 08: military_tank        conf=0.854 GT-IoU=0.869
Pass 09: military_tank        conf=0.803 GT-IoU=0.875
Pass 10: military_tank        conf=0.777 GT-IoU=0.944
Pass 11: military_tank        conf=0.784 GT-IoU=0.968
Pass 12: military_tank        conf=0.858 GT-IoU=0.909
Pass 13: military_tank        conf=0.832 GT-IoU=0.962
Pass 14: military_tank        conf=0.840 GT-IoU=0.943
Pass 15: military_tank        conf=0.840 GT-IoU=0.927
Pass 16: military_tank        conf=0.774 GT-IoU=0.900
Pass 17: military_tank        conf=0.827 GT-IoU=0.895
Pass 18: military_tank        conf=0.751 GT-IoU=0.885
Pass 19: military_tank      

In [22]:
from collections import Counter
import math
import numpy as np

detected_count = len(matched_detections)

persistence = detected_count / len(difficult_mc)

confidences = np.array([
    d["confidence"]
    for d in matched_detections
])

boxes = np.array([
    d["box"]
    for d in matched_detections
])

class_ids = [
    d["class_id"]
    for d in matched_detections
]

class_counts = Counter(class_ids)

dominant_class_id, dominant_count = \
    class_counts.most_common(1)[0]

dominant_class = v2_final.names[
    dominant_class_id
]

class_agreement = (
    dominant_count / detected_count
    if detected_count else 0
)

entropy = 0.0

for count in class_counts.values():

    p = count / detected_count
    entropy -= p * math.log2(p)


centers_x = (
    boxes[:, 0] + boxes[:, 2]
) / 2

centers_y = (
    boxes[:, 1] + boxes[:, 3]
) / 2

widths = boxes[:, 2] - boxes[:, 0]
heights = boxes[:, 3] - boxes[:, 1]

gt_ious = np.array([
    d["gt_iou"]
    for d in matched_detections
])


print("\n" + "=" * 60)
print("CORRECTED CLASS-AGNOSTIC MC-DROPOUT METRICS")
print("=" * 60)

print("Dominant class:       ", dominant_class)
print(f"Detected:              {detected_count}/{len(difficult_mc)}")
print(f"Persistence:           {persistence:.3f}")

print(f"Mean confidence:       {confidences.mean():.4f}")
print(f"Confidence std:        {confidences.std():.4f}")

print(f"Class agreement:       {class_agreement:.3f}")
print(f"Class entropy:         {entropy:.4f} bits")

print(f"Center X std:          {centers_x.std():.2f} px")
print(f"Center Y std:          {centers_y.std():.2f} px")

print(f"Width std:             {widths.std():.2f} px")
print(f"Height std:            {heights.std():.2f} px")

print(f"Mean GT IoU:           {gt_ious.mean():.4f}")
print(f"Minimum GT IoU:        {gt_ious.min():.4f}")

print("\nClass distribution:")

for cls_id, count in class_counts.items():

    print(
        f"  {v2_final.names[cls_id]}: "
        f"{count}/{len(difficult_mc)}"
    )


CORRECTED CLASS-AGNOSTIC MC-DROPOUT METRICS
Dominant class:        military_tank
Detected:              20/20
Persistence:           1.000
Mean confidence:       0.7917
Confidence std:        0.0678
Class agreement:       1.000
Class entropy:         0.0000 bits
Center X std:          3.38 px
Center Y std:          3.06 px
Width std:             7.81 px
Height std:            6.87 px
Mean GT IoU:           0.8964
Minimum GT IoU:        0.7521

Class distribution:
  military_tank: 20/20


In [23]:
import numpy as np
import cv2
import torch
from pathlib import Path
from collections import Counter
import math

N_SEARCH_PASSES = 10
MAX_IMAGES = 150
MATCH_IOU = 0.50
TANK_CLASS_ID = 2

search_results = []

for image_idx, image_path in enumerate(candidate_images[:MAX_IMAGES]):

    label_path = val_labels / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    original = cv2.imread(str(image_path))

    if original is None:
        continue

    orig_h, orig_w = original.shape[:2]

    # -------------------------
    # Ground-truth tank boxes
    # -------------------------
    gt_tanks = []

    for line in label_path.read_text().strip().splitlines():

        values = line.split()

        if not values:
            continue

        cls_id = int(float(values[0]))

        if cls_id != TANK_CLASS_ID:
            continue

        xc, yc, w, h = map(float, values[1:5])

        xc *= orig_w
        yc *= orig_h
        w *= orig_w
        h *= orig_h

        gt_tanks.append(
            np.array([
                xc - w / 2,
                yc - h / 2,
                xc + w / 2,
                yc + h / 2
            ], dtype=float)
        )

    # Keep first experiment simple:
    # only images containing exactly one GT tank
    if len(gt_tanks) != 1:
        continue

    # -------------------------
    # Convert GT box to 640 space
    # -------------------------
    scale = min(
        IMGSZ / orig_w,
        IMGSZ / orig_h
    )

    new_w = round(orig_w * scale)
    new_h = round(orig_h * scale)

    pad_x = (IMGSZ - new_w) / 2
    pad_y = (IMGSZ - new_h) / 2

    gt = gt_tanks[0]

    gt_640 = np.array([
        gt[0] * scale + pad_x,
        gt[1] * scale + pad_y,
        gt[2] * scale + pad_x,
        gt[3] * scale + pad_y
    ])

    # -------------------------
    # MC inference
    # -------------------------
    mc_results = mc_detect_image(
        image_path,
        n_passes=N_SEARCH_PASSES
    )

    matched = []

    for pass_idx, detections in enumerate(mc_results):

        best_det = None
        best_iou = 0.0

        # CLASS-AGNOSTIC matching
        for det in detections:

            iou = box_iou(
                det["box"],
                gt_640
            )

            if iou > best_iou:
                best_iou = iou
                best_det = det

        if (
            best_det is not None
            and best_iou >= MATCH_IOU
        ):
            matched.append({
                **best_det,
                "gt_iou": best_iou
            })

    detected = len(matched)

    persistence = (
        detected / N_SEARCH_PASSES
    )

    if detected == 0:
        search_results.append({
            "image": image_path,
            "persistence": 0.0,
            "class_agreement": 0.0,
            "entropy": 0.0,
            "conf_std": None,
            "mean_gt_iou": None,
            "min_gt_iou": None
        })

        continue

    classes = [
        d["class_id"]
        for d in matched
    ]

    confidences = np.array([
        d["confidence"]
        for d in matched
    ])

    gt_ious = np.array([
        d["gt_iou"]
        for d in matched
    ])

    counts = Counter(classes)

    dominant_id, dominant_count = \
        counts.most_common(1)[0]

    class_agreement = (
        dominant_count / detected
    )

    entropy = 0.0

    for count in counts.values():
        p = count / detected
        entropy -= p * math.log2(p)

    search_results.append({
        "image": image_path,
        "persistence": persistence,
        "class_agreement": class_agreement,
        "entropy": entropy,
        "conf_std": float(confidences.std()),
        "mean_gt_iou": float(gt_ious.mean()),
        "min_gt_iou": float(gt_ious.min()),
        "classes": dict(counts)
    })

    print(
        f"{image_idx+1:03d} | "
        f"P={persistence:.2f} | "
        f"A={class_agreement:.2f} | "
        f"H={entropy:.3f} | "
        f"Cstd={confidences.std():.3f} | "
        f"IoU={gt_ious.mean():.3f} | "
        f"{image_path.name}"
    )

001 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.051 | IoU=0.940 | 013256.jpg
002 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.073 | IoU=0.919 | 008516.jpg
003 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.065 | IoU=0.940 | 011600.jpg
007 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.048 | IoU=0.866 | 011399.jpg
008 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.084 | IoU=0.887 | 016696.jpg
009 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.051 | IoU=0.867 | 018582.jpg
011 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.030 | IoU=0.916 | 011357.jpg
012 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.054 | IoU=0.895 | 013339.jpg
013 | P=0.10 | A=1.00 | H=0.000 | Cstd=0.000 | IoU=0.504 | 016876.jpg
016 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.084 | IoU=0.696 | 016877.jpg
017 | P=1.00 | A=0.70 | H=0.881 | Cstd=0.140 | IoU=0.939 | 011333.jpg
018 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.152 | IoU=0.881 | 016997.jpg
019 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.081 | IoU=0.832 | 016952.jpg
020 | P=1.00 | A=1.00 | H=0.000 | Cstd=0.057 | IoU=0.815 | 011588.jpg
022 | P=1.00 | A=1.0

In [24]:
interesting = [
    r for r in search_results
    if (
        r["persistence"] < 1.0
        or r["class_agreement"] < 1.0
        or r["entropy"] > 0
        or (
            r["conf_std"] is not None
            and r["conf_std"] > 0.10
        )
        or (
            r["min_gt_iou"] is not None
            and r["min_gt_iou"] < 0.70
        )
    )
]

print("\n" + "=" * 70)
print("MOST INTERESTING UNCERTAIN CASES")
print("=" * 70)

for r in interesting[:20]:

    print(
        f'{r["image"].name:<15} '
        f'P={r["persistence"]:.2f}  '
        f'A={r["class_agreement"]:.2f}  '
        f'H={r["entropy"]:.3f}  '
        f'Cstd={r["conf_std"]}  '
        f'MinIoU={r["min_gt_iou"]}'
    )


MOST INTERESTING UNCERTAIN CASES
018582.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.05066025712634222  MinIoU=0.6562817445560706
016876.jpg      P=0.10  A=1.00  H=0.000  Cstd=0.0  MinIoU=0.5042450721836426
016877.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.08420776920187482  MinIoU=0.661367840426046
011333.jpg      P=1.00  A=0.70  H=0.881  Cstd=0.13963089278362417  MinIoU=0.8844815663994917
016997.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.15186918887308817  MinIoU=0.8605497785077539
016952.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.0809679299080873  MinIoU=0.633825922523207
011588.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.056772154517533986  MinIoU=0.6238183309742426
013334.jpg      P=0.90  A=1.00  H=0.000  Cstd=0.08077137028253732  MinIoU=0.7615422502041401
008633.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.10206694516370907  MinIoU=0.7559500733210474
013299.jpg      P=0.00  A=0.00  H=0.000  Cstd=None  MinIoU=None
018702.jpg      P=1.00  A=1.00  H=0.000  Cstd=0.11611333934447633  MinIoU=0.7713648

In [25]:
UNCERTAIN_IMAGE = (
    val_images / "011596.jpg"
)

uncertain_mc = mc_detect_image(
    UNCERTAIN_IMAGE,
    n_passes=20
)

print("Image:", UNCERTAIN_IMAGE)
print("MC passes:", len(uncertain_mc))

Image: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset/val/images/011596.jpg
MC passes: 20


In [26]:
label_path = (
    val_labels /
    f"{UNCERTAIN_IMAGE.stem}.txt"
)

original = cv2.imread(str(UNCERTAIN_IMAGE))
orig_h, orig_w = original.shape[:2]

gt_tanks = []

for line in label_path.read_text().strip().splitlines():

    values = line.split()

    if not values:
        continue

    cls_id = int(float(values[0]))

    if cls_id != TANK_CLASS_ID:
        continue

    xc, yc, w, h = map(float, values[1:5])

    xc *= orig_w
    yc *= orig_h
    w *= orig_w
    h *= orig_h

    gt_tanks.append(
        np.array([
            xc - w/2,
            yc - h/2,
            xc + w/2,
            yc + h/2
        ], dtype=float)
    )

print("GT tanks:", len(gt_tanks))

GT tanks: 1


In [27]:
scale = min(
    IMGSZ / orig_w,
    IMGSZ / orig_h
)

new_w = round(orig_w * scale)
new_h = round(orig_h * scale)

pad_x = (IMGSZ - new_w) / 2
pad_y = (IMGSZ - new_h) / 2

gt = gt_tanks[0]

GT_BOX = np.array([
    gt[0] * scale + pad_x,
    gt[1] * scale + pad_y,
    gt[2] * scale + pad_x,
    gt[3] * scale + pad_y
])

print("GT:", np.round(GT_BOX, 2))

GT: [      87.47       314.7       270.4      430.97]


In [28]:
MATCH_IOU = 0.50

matched = []

for pass_idx, detections in enumerate(uncertain_mc):

    best_det = None
    best_iou = 0.0

    for det in detections:

        iou = box_iou(
            det["box"],
            GT_BOX
        )

        if iou > best_iou:
            best_iou = iou
            best_det = det

    if (
        best_det is not None
        and best_iou >= MATCH_IOU
    ):

        matched.append({
            **best_det,
            "pass_idx": pass_idx,
            "gt_iou": best_iou
        })

        print(
            f'Pass {pass_idx+1:02d}: '
            f'{best_det["class_name"]:<20} '
            f'conf={best_det["confidence"]:.3f} '
            f'IoU={best_iou:.3f}'
        )

    else:

        print(
            f"Pass {pass_idx+1:02d}: MISSED"
        )

Pass 01: military_vehicle     conf=0.637 IoU=0.945
Pass 02: military_vehicle     conf=0.573 IoU=0.905
Pass 03: military_vehicle     conf=0.589 IoU=0.946
Pass 04: military_vehicle     conf=0.711 IoU=0.955
Pass 05: military_tank        conf=0.488 IoU=0.942
Pass 06: military_tank        conf=0.455 IoU=0.900
Pass 07: military_tank        conf=0.497 IoU=0.926
Pass 08: military_vehicle     conf=0.585 IoU=0.956
Pass 09: military_vehicle     conf=0.324 IoU=0.932
Pass 10: military_vehicle     conf=0.693 IoU=0.967
Pass 11: military_tank        conf=0.541 IoU=0.902
Pass 12: military_vehicle     conf=0.651 IoU=0.925
Pass 13: military_vehicle     conf=0.645 IoU=0.942
Pass 14: military_tank        conf=0.436 IoU=0.923
Pass 15: military_tank        conf=0.470 IoU=0.923
Pass 16: military_tank        conf=0.412 IoU=0.935
Pass 17: military_vehicle     conf=0.578 IoU=0.926
Pass 18: military_tank        conf=0.437 IoU=0.960
Pass 19: military_vehicle     conf=0.381 IoU=0.868
Pass 20: military_vehicle     c

In [29]:
from collections import Counter
import math
import numpy as np

N_MC = 20
detected = len(matched)

persistence = detected / N_MC

if detected > 0:

    confidences = np.array([
        d["confidence"]
        for d in matched
    ])

    boxes = np.array([
        d["box"]
        for d in matched
    ])

    class_ids = [
        d["class_id"]
        for d in matched
    ]

    gt_ious = np.array([
        d["gt_iou"]
        for d in matched
    ])

    counts = Counter(class_ids)

    dominant_id, dominant_count = \
        counts.most_common(1)[0]

    class_agreement = (
        dominant_count / detected
    )

    entropy = 0.0

    for count in counts.values():

        p = count / detected
        entropy -= p * math.log2(p)

    centers_x = (
        boxes[:, 0] + boxes[:, 2]
    ) / 2

    centers_y = (
        boxes[:, 1] + boxes[:, 3]
    ) / 2

    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]

    print("\n" + "="*60)
    print("UNCERTAIN TARGET — FULL 20-PASS MCDO")
    print("="*60)

    print("Image:              ", UNCERTAIN_IMAGE.name)

    print(
        "Dominant class:     ",
        v2_final.names[dominant_id]
    )

    print(f"Detected:            {detected}/{N_MC}")
    print(f"Persistence:         {persistence:.3f}")

    print(f"Mean confidence:     {confidences.mean():.4f}")
    print(f"Confidence std:      {confidences.std():.4f}")

    print(f"Class agreement:     {class_agreement:.3f}")
    print(f"Class entropy:       {entropy:.4f} bits")

    print(f"Center X std:        {centers_x.std():.2f} px")
    print(f"Center Y std:        {centers_y.std():.2f} px")

    print(f"Width std:           {widths.std():.2f} px")
    print(f"Height std:          {heights.std():.2f} px")

    print(f"Mean GT IoU:         {gt_ious.mean():.4f}")
    print(f"Minimum GT IoU:      {gt_ious.min():.4f}")

    print("\nClass distribution:")

    for cls_id, count in counts.items():

        print(
            f"  {v2_final.names[cls_id]}: "
            f"{count}/{N_MC}"
        )

else:

    print("Target never detected.")


UNCERTAIN TARGET — FULL 20-PASS MCDO
Image:               011596.jpg
Dominant class:      military_vehicle
Detected:            20/20
Persistence:         1.000
Mean confidence:     0.5200
Confidence std:      0.1163
Class agreement:     0.600
Class entropy:       0.9710 bits
Center X std:        2.07 px
Center Y std:        2.05 px
Width std:           2.67 px
Height std:          4.11 px
Mean GT IoU:         0.9312
Minimum GT IoU:      0.8684

Class distribution:
  military_vehicle: 12/20
  military_tank: 8/20


In [30]:
from ultralytics import YOLO
from pathlib import Path

MODEL_V1 = (
    "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/"
    "baseline/military_kaggle_v1.pt"
)

v1_model = YOLO(MODEL_V1)

v1_result = v1_model.predict(
    source=str(UNCERTAIN_IMAGE),
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)[0]

print("=" * 55)
print("V1 DETERMINISTIC RESULT — 011596.jpg")
print("=" * 55)

if v1_result.boxes is None or len(v1_result.boxes) == 0:
    print("No detections")

else:
    for i, box in enumerate(v1_result.boxes, start=1):

        cls_id = int(box.cls.item())
        conf = float(box.conf.item())

        print(
            f"Detection {i}: "
            f"{v1_model.names[cls_id]} "
            f"| confidence={conf:.4f}"
        )

V1 DETERMINISTIC RESULT — 011596.jpg
Detection 1: military_vehicle | confidence=0.7968
Detection 2: military_tank | confidence=0.6137


In [31]:
import numpy as np

print("=" * 60)
print("V1 BOX MATCHING — 011596.jpg")
print("=" * 60)

v1_boxes_info = []

for i, box in enumerate(v1_result.boxes, start=1):

    cls_id = int(box.cls.item())
    conf = float(box.conf.item())

    # xyxy coordinates produced by V1
    xyxy = box.xyxy[0].detach().cpu().numpy().astype(float)

    iou_gt = box_iou(
        xyxy,
        GT_BOX
    )

    v1_boxes_info.append({
        "index": i,
        "class_id": cls_id,
        "class_name": v1_model.names[cls_id],
        "confidence": conf,
        "box": xyxy,
        "gt_iou": iou_gt
    })

    print(
        f'Detection {i}: '
        f'{v1_model.names[cls_id]:<20} '
        f'conf={conf:.4f} '
        f'GT-IoU={iou_gt:.4f}'
    )

V1 BOX MATCHING — 011596.jpg
Detection 1: military_vehicle     conf=0.7968 GT-IoU=0.0000
Detection 2: military_tank        conf=0.6137 GT-IoU=0.0000


In [32]:
if len(v1_boxes_info) >= 2:

    overlap = box_iou(
        v1_boxes_info[0]["box"],
        v1_boxes_info[1]["box"]
    )

    print(
        "\nIoU between V1 detection 1 and 2:",
        f"{overlap:.4f}"
    )


IoU between V1 detection 1 and 2: 0.9856


In [33]:
GT_BOX_ORIGINAL = gt_tanks[0]

print("=" * 60)
print("CORRECTED V1 BOX MATCHING — 011596.jpg")
print("=" * 60)

v1_boxes_info = []

for i, box in enumerate(v1_result.boxes, start=1):

    cls_id = int(box.cls.item())
    conf = float(box.conf.item())

    xyxy = (
        box.xyxy[0]
        .detach()
        .cpu()
        .numpy()
        .astype(float)
    )

    iou_gt = box_iou(
        xyxy,
        GT_BOX_ORIGINAL
    )

    v1_boxes_info.append({
        "index": i,
        "class_id": cls_id,
        "class_name": v1_model.names[cls_id],
        "confidence": conf,
        "box": xyxy,
        "gt_iou": iou_gt
    })

    print(
        f'Detection {i}: '
        f'{v1_model.names[cls_id]:<20} '
        f'conf={conf:.4f} '
        f'GT-IoU={iou_gt:.4f}'
    )

if len(v1_boxes_info) >= 2:

    overlap = box_iou(
        v1_boxes_info[0]["box"],
        v1_boxes_info[1]["box"]
    )

    print(
        "\nIoU between V1 detections:",
        f"{overlap:.4f}"
    )

CORRECTED V1 BOX MATCHING — 011596.jpg
Detection 1: military_vehicle     conf=0.7968 GT-IoU=0.9328
Detection 2: military_tank        conf=0.6137 GT-IoU=0.9331

IoU between V1 detections: 0.9856


In [34]:
print("=" * 65)
print("V2 OVERLAPPING PREDICTIONS PER MC PASS")
print("=" * 65)

for pass_idx, detections in enumerate(uncertain_mc, start=1):

    overlapping = []

    for det in detections:

        iou = box_iou(
            det["box"],
            GT_BOX
        )

        if iou >= 0.50:
            overlapping.append(
                (
                    det["class_name"],
                    det["confidence"],
                    iou
                )
            )

    print(f"\nPass {pass_idx:02d}:")

    if not overlapping:
        print("  MISSED")

    else:
        for cls_name, conf, iou in overlapping:
            print(
                f"  {cls_name:<20} "
                f"conf={conf:.3f} "
                f"GT-IoU={iou:.3f}"
            )

V2 OVERLAPPING PREDICTIONS PER MC PASS

Pass 01:
  military_vehicle     conf=0.637 GT-IoU=0.945
  military_tank        conf=0.489 GT-IoU=0.654

Pass 02:
  military_vehicle     conf=0.573 GT-IoU=0.905
  military_tank        conf=0.471 GT-IoU=0.845

Pass 03:
  military_vehicle     conf=0.589 GT-IoU=0.946

Pass 04:
  military_vehicle     conf=0.711 GT-IoU=0.955
  military_tank        conf=0.492 GT-IoU=0.907

Pass 05:
  military_vehicle     conf=0.514 GT-IoU=0.910
  military_tank        conf=0.488 GT-IoU=0.942

Pass 06:
  military_vehicle     conf=0.497 GT-IoU=0.888
  military_tank        conf=0.455 GT-IoU=0.900

Pass 07:
  military_tank        conf=0.497 GT-IoU=0.926
  military_vehicle     conf=0.290 GT-IoU=0.917

Pass 08:
  military_vehicle     conf=0.585 GT-IoU=0.956
  military_tank        conf=0.530 GT-IoU=0.836

Pass 09:
  military_tank        conf=0.621 GT-IoU=0.852
  military_vehicle     conf=0.324 GT-IoU=0.932

Pass 10:
  military_vehicle     conf=0.693 GT-IoU=0.967
  military_tank

In [35]:
from collections import Counter, defaultdict
import numpy as np
import math

MATCH_IOU = 0.50
N_MC = len(uncertain_mc)

pass_records = []

for pass_idx, detections in enumerate(uncertain_mc):

    # ------------------------------------
    # Collect ALL detections for this object
    # regardless of predicted class
    # ------------------------------------
    matched = []

    for det in detections:

        iou = box_iou(
            det["box"],
            GT_BOX
        )

        if iou >= MATCH_IOU:
            matched.append({
                **det,
                "gt_iou": iou
            })

    if not matched:

        pass_records.append({
            "pass": pass_idx + 1,
            "detected": False
        })

        continue

    # ------------------------------------
    # Keep strongest hypothesis PER CLASS
    # ------------------------------------
    by_class = {}

    for det in matched:

        cid = det["class_id"]

        if (
            cid not in by_class
            or det["confidence"] >
               by_class[cid]["confidence"]
        ):
            by_class[cid] = det

    class_hypotheses = list(by_class.values())

    # ------------------------------------
    # Winner = highest confidence class
    # NOT highest IoU
    # ------------------------------------
    winner = max(
        class_hypotheses,
        key=lambda d: d["confidence"]
    )

    # ------------------------------------
    # Normalize confidence evidence
    #
    # IMPORTANT:
    # this is a useful evidence summary,
    # NOT a calibrated probability.
    # ------------------------------------
    conf_sum = sum(
        d["confidence"]
        for d in class_hypotheses
    )

    evidence = {
        d["class_id"]:
        d["confidence"] / conf_sum
        for d in class_hypotheses
    }

    # ------------------------------------
    # Confidence-weighted representative box
    # so class choice does not determine
    # localization measurement
    # ------------------------------------
    boxes_local = np.array([
        d["box"]
        for d in class_hypotheses
    ])

    weights = np.array([
        d["confidence"]
        for d in class_hypotheses
    ])

    representative_box = np.average(
        boxes_local,
        axis=0,
        weights=weights
    )

    rep_gt_iou = box_iou(
        representative_box,
        GT_BOX
    )

    pass_records.append({

        "pass": pass_idx + 1,
        "detected": True,

        "winner_id": winner["class_id"],
        "winner_name": winner["class_name"],
        "winner_confidence": winner["confidence"],

        "num_class_hypotheses":
            len(class_hypotheses),

        "classes": [
            d["class_name"]
            for d in class_hypotheses
        ],

        "evidence": evidence,

        "representative_box":
            representative_box,

        "gt_iou":
            rep_gt_iou
    })

In [36]:
print("=" * 70)
print("PER-PASS CLASS COMPETITION")
print("=" * 70)

for r in pass_records:

    if not r["detected"]:

        print(
            f'Pass {r["pass"]:02d}: MISSED'
        )

        continue

    competing = (
        "YES"
        if r["num_class_hypotheses"] > 1
        else "NO"
    )

    print(
        f'Pass {r["pass"]:02d}: '
        f'winner={r["winner_name"]:<18} '
        f'conf={r["winner_confidence"]:.3f} | '
        f'classes={r["classes"]} | '
        f'competition={competing}'
    )

PER-PASS CLASS COMPETITION
Pass 01: winner=military_vehicle   conf=0.637 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 02: winner=military_vehicle   conf=0.573 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 03: winner=military_vehicle   conf=0.589 | classes=['military_vehicle'] | competition=NO
Pass 04: winner=military_vehicle   conf=0.711 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 05: winner=military_vehicle   conf=0.514 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 06: winner=military_vehicle   conf=0.497 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 07: winner=military_tank      conf=0.497 | classes=['military_tank', 'military_vehicle'] | competition=YES
Pass 08: winner=military_vehicle   conf=0.585 | classes=['military_vehicle', 'military_tank'] | competition=YES
Pass 09: winner=military_tank      conf=0.621 | classes=['military_tank', 'military_vehicle'] |

In [37]:
detected_records = [
    r for r in pass_records
    if r["detected"]
]

detected_count = len(detected_records)

persistence = (
    detected_count / N_MC
)

# ====================================
# 1. WINNER DISTRIBUTION
# ====================================

winner_ids = [
    r["winner_id"]
    for r in detected_records
]

winner_counts = Counter(winner_ids)

dominant_id, dominant_count = \
    winner_counts.most_common(1)[0]

winner_agreement = (
    dominant_count / detected_count
)

winner_entropy = 0.0

for count in winner_counts.values():

    p = count / detected_count

    winner_entropy -= (
        p * math.log2(p)
    )


# ====================================
# 2. WITHIN-PASS CLASS COMPETITION
# ====================================

competing_passes = sum(
    r["num_class_hypotheses"] > 1
    for r in detected_records
)

competition_rate = (
    competing_passes / detected_count
)


# ====================================
# 3. NORMALIZED CLASS EVIDENCE
# ====================================

all_class_ids = sorted({
    cid
    for r in detected_records
    for cid in r["evidence"]
})

mean_evidence = {}

for cid in all_class_ids:

    mean_evidence[cid] = np.mean([
        r["evidence"].get(cid, 0.0)
        for r in detected_records
    ])

evidence_entropy = 0.0

for p in mean_evidence.values():

    if p > 0:
        evidence_entropy -= (
            p * math.log2(p)
        )


# ====================================
# 4. WINNER CONFIDENCE VARIABILITY
# ====================================

winner_confidences = np.array([
    r["winner_confidence"]
    for r in detected_records
])


# ====================================
# 5. LOCALIZATION
# ====================================

rep_boxes = np.array([
    r["representative_box"]
    for r in detected_records
])

centers_x = (
    rep_boxes[:, 0] +
    rep_boxes[:, 2]
) / 2

centers_y = (
    rep_boxes[:, 1] +
    rep_boxes[:, 3]
) / 2

widths = (
    rep_boxes[:, 2] -
    rep_boxes[:, 0]
)

heights = (
    rep_boxes[:, 3] -
    rep_boxes[:, 1]
)

gt_ious = np.array([
    r["gt_iou"]
    for r in detected_records
])

In [38]:
print("\n" + "=" * 65)
print("REFINED V2 MC-DROPOUT UNCERTAINTY — 011596.jpg")
print("=" * 65)

print(
    "Dominant winner:       ",
    v2_final.names[dominant_id]
)

print(
    f"Detected:               "
    f"{detected_count}/{N_MC}"
)

print(
    f"Persistence:            "
    f"{persistence:.3f}"
)

print("\n--- Across-pass class uncertainty ---")

print(
    f"Winner agreement:       "
    f"{winner_agreement:.3f}"
)

print(
    f"Winner entropy:         "
    f"{winner_entropy:.4f} bits"
)

print("\nWinner distribution:")

for cid, count in winner_counts.items():

    print(
        f"  {v2_final.names[cid]}: "
        f"{count}/{detected_count}"
    )


print("\n--- Within-pass ambiguity ---")

print(
    f"Competing-class passes: "
    f"{competing_passes}/{detected_count}"
)

print(
    f"Competition rate:       "
    f"{competition_rate:.3f}"
)


print("\n--- Mean normalized class evidence ---")

for cid, evidence in mean_evidence.items():

    print(
        f"  {v2_final.names[cid]}: "
        f"{evidence:.3f}"
    )

print(
    f"Evidence entropy:       "
    f"{evidence_entropy:.4f} bits"
)


print("\n--- Confidence ---")

print(
    f"Winner confidence mean: "
    f"{winner_confidences.mean():.4f}"
)

print(
    f"Winner confidence std:  "
    f"{winner_confidences.std():.4f}"
)


print("\n--- Localization ---")

print(
    f"Center X std:           "
    f"{centers_x.std():.2f} px"
)

print(
    f"Center Y std:           "
    f"{centers_y.std():.2f} px"
)

print(
    f"Width std:              "
    f"{widths.std():.2f} px"
)

print(
    f"Height std:             "
    f"{heights.std():.2f} px"
)

print(
    f"Mean GT IoU:            "
    f"{gt_ious.mean():.4f}"
)

print(
    f"Minimum GT IoU:         "
    f"{gt_ious.min():.4f}"
)


REFINED V2 MC-DROPOUT UNCERTAINTY — 011596.jpg
Dominant winner:        military_vehicle
Detected:               20/20
Persistence:            1.000

--- Across-pass class uncertainty ---
Winner agreement:       0.650
Winner entropy:         0.9341 bits

Winner distribution:
  military_vehicle: 13/20
  military_tank: 7/20

--- Within-pass ambiguity ---
Competing-class passes: 18/20
Competition rate:       0.900

--- Mean normalized class evidence ---
  military_tank: 0.486
  military_vehicle: 0.514
Evidence entropy:       0.9995 bits

--- Confidence ---
Winner confidence mean: 0.5659
Winner confidence std:  0.1016

--- Localization ---
Center X std:           6.24 px
Center Y std:           2.15 px
Width std:              13.00 px
Height std:             3.22 px
Mean GT IoU:            0.9092
Minimum GT IoU:         0.7702


In [39]:
import json
from pathlib import Path

RESULT_DIR = Path(
    "/content/drive/MyDrive/UAV_MC_DROPOUT_V2/"
    "headonly_p020_lr1e4_e3/experiments"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)

result = {
    "image": "011596.jpg",

    "v1": {
        "detections": [
            {
                "class": "military_vehicle",
                "confidence": 0.7968,
                "gt_iou": 0.9328
            },
            {
                "class": "military_tank",
                "confidence": 0.6137,
                "gt_iou": 0.9331
            }
        ],
        "detection_box_iou": 0.9856
    },

    "v2_mc_dropout": {
        "passes": 20,
        "persistence": 1.0,

        "winner_distribution": {
            "military_vehicle": 13,
            "military_tank": 7
        },

        "winner_agreement": 0.650,
        "winner_entropy_bits": 0.9341,

        "competing_class_passes": 18,
        "competition_rate": 0.900,

        "mean_normalized_evidence": {
            "military_vehicle": 0.514,
            "military_tank": 0.486
        },

        "evidence_entropy_bits": 0.9995,

        "winner_confidence_mean": 0.5659,
        "winner_confidence_std": 0.1016,

        "center_x_std_px": 6.24,
        "center_y_std_px": 2.15,
        "width_std_px": 13.00,
        "height_std_px": 3.22,

        "mean_gt_iou": 0.9092,
        "minimum_gt_iou": 0.7702
    }
}

SAVE_FILE = RESULT_DIR / "011596_v1_vs_v2_uncertainty.json"

with open(SAVE_FILE, "w") as f:
    json.dump(result, f, indent=2)

print("Saved:", SAVE_FILE)

Saved: /content/drive/MyDrive/UAV_MC_DROPOUT_V2/headonly_p020_lr1e4_e3/experiments/011596_v1_vs_v2_uncertainty.json
